# 03. EDA, estadisticas y visualizaciones

Esta notebook integra el analisis descriptivo, la lectura estadistica y las visualizaciones principales del proyecto sobre una sola fuente final: `survey_data_cleaned_reduced.csv`.

## Alcance

- cargar el dataset final limpio y reducido
- revisar estructura, calidad final y variables clave
- resumir estadisticamente salario, experiencia y satisfaccion
- perfilar segmentos demograficos y laborales
- analizar tendencias actuales y futuras de tecnologias
- visualizar los hallazgos dentro de la misma notebook
- cerrar con una lectura sintetica de correlaciones y patrones

**Nota:** esta notebook no exporta archivos. Todo el analisis tabular y grafico queda integrado aqui para facilitar lectura y comprension.


## Subpaso 1. Carga del dataset final y helpers

**Proposito:** preparar un entorno unico para el analisis estadistico y visual.

**Resultado esperado:** acceso a `data/survey_data_cleaned_reduced.csv` y funciones auxiliares para trabajar con variables multiseleccion y grupos etarios.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "survey_data_cleaned_reduced.csv"

df = pd.read_csv(INPUT_PATH)

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

AGE_ORDER = [
    "Under 18 years old",
    "18-24 years old",
    "25-34 years old",
    "35-44 years old",
    "45-54 years old",
    "55-64 years old",
    "65 years or older",
    "Prefer not to say",
]

AGE_MID_MAP = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29.5,
    "35-44 years old": 39.5,
    "45-54 years old": 49.5,
    "55-64 years old": 59.5,
    "65 years or older": 70,
    "Prefer not to say": None,
}

def split_multiselect(series: pd.Series) -> pd.Series:
    cleaned = series.dropna().astype(str).str.strip()
    cleaned = cleaned[~cleaned.isin(["", "Not specified"])]
    return cleaned.str.split(";").explode().str.strip()

def top_multiselect_counts(series: pd.Series, top_n: int = 10, label: str = "item") -> pd.DataFrame:
    counts = split_multiselect(series).value_counts().head(top_n)
    return counts.rename_axis(label).reset_index(name="count")

INPUT_PATH


## Subpaso 2. Vista estructural del dataset

**Proposito:** confirmar que la version final realmente quede lista para analisis sin pasos intermedios adicionales.

**Resultado:** se valida estructura, cantidad de filas, columnas y ausencia de faltantes.

**Hallazgos:** el dataset final mantiene el foco en demografia, experiencia, salario, satisfaccion y stack tecnologico, sin arrastrar columnas de bajo aporte para la narrativa del proyecto.


In [ ]:
structure_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null_count": df.count(),
        "missing_values": df.isna().sum(),
    }
)

print("Shape:", df.shape)
print("Total missing values:", int(df.isna().sum().sum()))
structure_summary


## Subpaso 3. Compensacion, experiencia y satisfaccion

**Proposito:** leer primero la base numerica del caso y luego verla representada visualmente.

**Resultado:** se resume `ConvertedCompYearly`, `WorkExp`, `JobSat` y `JobSatPoints_6`, y se visualiza la distribucion salarial principal.

**Hallazgos:** la compensacion anual presenta alta dispersion y una cola larga; la experiencia laboral tiene una mediana cercana a `10` anos; la satisfaccion laboral se concentra en valores relativamente altos.


In [ ]:
numeric_columns = [
    "CompTotal",
    "ConvertedCompYearly",
    "WorkExp",
    "JobSat",
    "JobSatPoints_6",
    "ConvertedCompYearly_MinMax",
    "ConvertedCompYearly_Zscore",
]

numeric_summary = df[numeric_columns].describe().transpose()
numeric_summary


In [ ]:
salary_vis = df["ConvertedCompYearly"].clip(upper=df["ConvertedCompYearly"].quantile(0.99))

plt.figure(figsize=(10, 5))
plt.hist(salary_vis, bins=30, color="#4C78A8", alpha=0.85)
plt.title("Distribution of Converted Annual Compensation (trimmed at p99)")
plt.xlabel("ConvertedCompYearly")
plt.ylabel("Respondent count")
plt.tight_layout()
plt.show()


## Subpaso 4. Perfil demografico y laboral

**Proposito:** entender quienes componen la muestra antes de pasar a comparaciones mas especificas.

**Resultado:** se construyen tablas de frecuencia para edad, empleo, modalidad de trabajo, pais, tipo de desarrollador e industria.

**Hallazgos:** `25-34 years old` es el grupo dominante; el empleo full-time concentra gran parte de la muestra; `Remote` y `Hybrid` lideran entre modalidades; `United States`, `Germany`, `India` y `United Kingdom` destacan entre paises con mas respuestas.


In [ ]:
age_distribution = df["Age"].value_counts().reindex(AGE_ORDER, fill_value=0).rename_axis("Age").reset_index(name="count")
employment_distribution = df["Employment"].value_counts().rename_axis("Employment").reset_index(name="count")
remotework_distribution = df["RemoteWork"].value_counts().rename_axis("RemoteWork").reset_index(name="count")
country_top10 = df["Country"].value_counts().head(10).rename_axis("Country").reset_index(name="count")
devtype_top10 = split_multiselect(df["DevType"]).value_counts().head(10).rename_axis("DevType").reset_index(name="count")
industry_top10 = df["Industry"].value_counts().head(10).rename_axis("Industry").reset_index(name="count")

display(age_distribution)
display(remotework_distribution)
display(country_top10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(age_distribution["Age"], age_distribution["count"], color="#76B7B2")
axes[0].set_title("Respondent Distribution by Age Group")
axes[0].set_xlabel("")
axes[0].set_ylabel("Respondent count")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(remotework_distribution["RemoteWork"], remotework_distribution["count"], color=["#E15759", "#59A14F", "#F28E2B"])
axes[1].set_title("Remote Work Distribution")
axes[1].set_xlabel("")
axes[1].set_ylabel("Respondent count")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## Subpaso 5. Edad y experiencia laboral

**Proposito:** conectar una lectura estadistica simple con una vista visual de trayectoria profesional.

**Resultado:** se resumen medias y medianas de experiencia por grupo etario y luego se visualiza la relacion entre edad y `WorkExp`.

**Hallazgos:** la experiencia tiende a crecer con la edad, pero con bastante dispersion dentro de cada tramo, lo que sugiere trayectorias profesionales heterogeneas.


In [ ]:
workexp_by_age = (
    df.groupby("Age")["WorkExp"]
    .agg(["count", "median", "mean"])
    .reindex(AGE_ORDER)
    .reset_index()
)
workexp_by_age


In [ ]:
scatter_df = df.assign(Age_num=df["Age"].map(AGE_MID_MAP))[["Age_num", "WorkExp"]].dropna()
if len(scatter_df) > 3000:
    scatter_df = scatter_df.sample(3000, random_state=42)

plt.figure(figsize=(9, 5))
plt.scatter(scatter_df["Age_num"], scatter_df["WorkExp"], alpha=0.35, color="#59A14F")
plt.title("Age vs Work Experience")
plt.xlabel("Age midpoint")
plt.ylabel("WorkExp")
plt.tight_layout()
plt.show()


## Subpaso 6. Comparaciones de compensacion por grupos

**Proposito:** comparar salario entre modalidades de trabajo, tipos de empleo y roles frecuentes.

**Resultado:** se combinan tablas de agregados con boxplots para mostrar tanto tendencia central como dispersion.

**Hallazgos:** `Remote` muestra la media salarial mas alta entre modalidades; las categorias de empleo y los perfiles de desarrollador mas frecuentes exhiben dispersion amplia, por lo que conviene leer medianas y rangos junto con promedios.


In [ ]:
salary_by_remotework = (
    df.groupby("RemoteWork")["ConvertedCompYearly"]
    .agg(["count", "median", "mean"])
    .sort_values("mean", ascending=False)
    .reset_index()
)

salary_by_age = (
    df.groupby("Age")["ConvertedCompYearly"]
    .agg(["count", "median", "mean"])
    .reindex(AGE_ORDER)
    .reset_index()
)

display(salary_by_remotework)
display(salary_by_age)


In [ ]:
employment_order = df["Employment"].value_counts().head(6).index.tolist()
employment_box = df[df["Employment"].isin(employment_order)][["Employment", "ConvertedCompYearly"]].dropna()
employment_groups = [employment_box.loc[employment_box["Employment"] == category, "ConvertedCompYearly"].values for category in employment_order]

plt.figure(figsize=(12, 6))
plt.boxplot(employment_groups, labels=employment_order, patch_artist=True)
plt.title("Converted Annual Compensation by Employment Type")
plt.xlabel("Employment")
plt.ylabel("ConvertedCompYearly")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
top_devtypes = split_multiselect(df["DevType"]).value_counts().head(5).index.tolist()
devtype_exploded = (
    df[["DevType", "ConvertedCompYearly"]]
    .dropna(subset=["DevType", "ConvertedCompYearly"])
    .assign(DevType=lambda d: d["DevType"].str.split(";"))
    .explode("DevType")
)
devtype_exploded["DevType"] = devtype_exploded["DevType"].str.strip()
devtype_box = devtype_exploded[devtype_exploded["DevType"].isin(top_devtypes)]
devtype_groups = [devtype_box.loc[devtype_box["DevType"] == category, "ConvertedCompYearly"].values for category in top_devtypes]

plt.figure(figsize=(12, 6))
plt.boxplot(devtype_groups, labels=top_devtypes, patch_artist=True)
plt.title("Converted Annual Compensation for Top Developer Types")
plt.xlabel("Developer type")
plt.ylabel("ConvertedCompYearly")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## Subpaso 7. Tecnologias actuales y de interes futuro

**Proposito:** comparar el stack reportado hoy con el stack que concentra interes para el proximo ano.

**Resultado:** se generan tablas Top 10 y dos comparaciones visuales dobles: lenguajes y bases de datos.

**Hallazgos:** `JavaScript`, `SQL`, `TypeScript` y `Python` mantienen presencia fuerte entre uso actual e interes futuro; `PostgreSQL` domina en bases de datos; tecnologias como `Go`, `Rust`, `Supabase` y `Redis` ayudan a contar la historia de momentum futuro.


In [ ]:
current_languages_top10 = top_multiselect_counts(df["LanguageHaveWorkedWith"], label="language")
future_languages_top10 = top_multiselect_counts(df["LanguageWantToWorkWith"], label="language")
current_databases_top10 = top_multiselect_counts(df["DatabaseHaveWorkedWith"], label="database")
future_databases_top10 = top_multiselect_counts(df["DatabaseWantToWorkWith"], label="database")

display(current_languages_top10)
display(future_languages_top10)
display(current_databases_top10)
display(future_databases_top10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(current_languages_top10["language"], current_languages_top10["count"], color="#4C78A8")
axes[0].set_title("Top 10 Programming Languages Used")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(future_languages_top10["language"], future_languages_top10["count"], color="#59A14F")
axes[1].set_title("Top 10 Programming Languages Desired Next Year")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(current_databases_top10["database"], current_databases_top10["count"], color="#F28E2B")
axes[0].set_title("Top 10 Databases Used")
axes[0].set_xlabel("")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=35)

axes[1].bar(future_databases_top10["database"], future_databases_top10["count"], color="#E15759")
axes[1].set_title("Top 10 Databases Desired Next Year")
axes[1].set_xlabel("")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()


## Subpaso 8. Correlacion numerica y cierre analitico

**Proposito:** cerrar la notebook con una lectura sintetica de relaciones lineales entre salario, experiencia y satisfaccion.

**Resultado:** se calcula una matriz de correlacion y se visualiza en un mapa de calor simple.

**Hallazgos:** `WorkExp` se relaciona de forma positiva pero moderada con `ConvertedCompYearly`; `JobSat` y `JobSatPoints_6` muestran asociaciones debiles con salario y experiencia, lo que sugiere que la satisfaccion no sigue una relacion lineal fuerte en este recorte.


In [ ]:
correlation_columns = [
    "CompTotal",
    "ConvertedCompYearly",
    "WorkExp",
    "JobSat",
    "JobSatPoints_6",
    "ConvertedCompYearly_MinMax",
    "ConvertedCompYearly_Zscore",
]

correlation_matrix = df[correlation_columns].corr(numeric_only=True)
correlation_matrix


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
heatmap = ax.imshow(correlation_matrix.values, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlation_matrix.index)))
ax.set_yticklabels(correlation_matrix.index)
ax.set_title("Correlation Matrix for Key Numeric Variables")

for row in range(len(correlation_matrix.index)):
    for col in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color="black", fontsize=8)

fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## Cierre de la etapa

Esta notebook deja integrado el recorrido analitico principal del proyecto: perfil de la muestra, variables numericas clave, comparaciones por grupos, tendencias tecnologicas y visualizaciones asociadas. Con esto, la siguiente notebook puede enfocarse solo en documentar el dashboard final en Cognos Analytics y la informacion que sintetiza.
